<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/07)_Mask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정

from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/attention_is_all_you_need"
)

SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print(PROJECT_ROOT)

/content/drive/MyDrive/attention_is_all_you_need


In [ ]:
# 기존 파일 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
'''
Mask는 2가지 종류로 나뉜다(Padding Mask, Casual Mask)

Padding Mask:
 문장의 길이를 맞추기 위해 덧붙인 의미없는 PAD 토큰을 계산에서 제외시키기 위해 사용.
 모든 입력 sequence에서 사용.
 PAD의 위치를 차단함

Casual Mask:
 현재 나올 단어를 예측 함에 있어 미래의 정보를 끌어와서 예측하지 않게 하기 위해 사용.
 Decoder의 self attention layer에서 사용.
 현재 위치보다 뒤에 나오는 정보들을 차단함


-> 둘 중 하나라도 차단하면 해당 위치는 차단됨

'''

In [ ]:
# 기존 Mask 처리 확인

print(
    (SRC_DIR / "attention.py").read_text()
)


import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class ScaledDotProductAttention(nn.Module):
    """
    Attention(Q, K, V)
        = softmax(QK^T / sqrt(d_k)) V
    """

    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q:
                (..., query_len, d_k)

            K:
                (..., key_len, d_k)

            V:
                (..., key_len, d_v)

            mask:
                Attention mask broadcastable to
                (..., query_len, key_len).

                True  = attention 허용
                False = attention 차단

        Returns:
            output:
                (..., query_len, d_v)

            attention_weights:
                (..., query_len, key_len)
        """

        if Q.size(-1) != K.size(-1):
            raise ValueError(
                f"Q와 K의 d_k가 같아야 합니다. "
                f"Q: {Q.size(-1)}, K: {K.size(-1)}"
          

In [ ]:
# MultiHeadAttention 구조 확인

print(
    (SRC_DIR / "encoder.py").read_text()
)


import torch.nn as nn

from src.encoder_layer import EncoderLayer


class Encoder(nn.Module):
    def __init__(
        self,
        d_model,
        num_heads,
        d_ff,
        num_layers,
        dropout=0.1,
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            EncoderLayer(
                d_model,
                num_heads,
                d_ff,
                dropout,
            )
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        attention_weights_list = []

        for layer in self.layers:
            x, attention_weights = layer(
                x,
                mask,
            )

            attention_weights_list.append(
                attention_weights
            )

        return x, attention_weights_list



In [ ]:
# mask.py 전체 코드

import torch


def create_padding_mask(
    token_ids,
    pad_idx,
):
    mask = token_ids != pad_idx # PAD를 False, 실제 데이터를 True로 둔다
    # 현재 shape는 (B, K) 이다.
    # Attention score는 (B, H, Q, K)의 shape을 가지기 때문에 두 차원을 추가해 주어야 한다.

    mask = mask.unsqueeze(1).unsqueeze(2)
    # 현재 shape는 (B, 1, 1, K)
    # Pytorch에서 크기가 1인 차원은 상대 Tensor 크기에 맞춰 Broadcasting 할 수 있기에 Attention score의 차원에 맞춰서 사용한다.
    # 따라서, 직접 repeat(직접 복사본을 만들어서 차원을 맞춰줌) 할 필요가 없다.

    return mask


def create_causal_mask(
    seq_len,
    device=None,
):
    mask = torch.ones(
        seq_len,
        seq_len,
        dtype=torch.bool,
        device=device,
    )

    mask = torch.tril(mask) # torch.trll은 하삼각 행렬(대각선에서 아랫쪽을 남기고 전부 0으로 바꿔줌)로 만들어주는 함수이다.

    mask = mask.unsqueeze(0).unsqueeze(0)

    # Attention score의 shape는 (B, H, Q, K)이다.
    # Decoder Self-Attention에서 Q = S(seq_len), K = S이다.
    # 따라서, (B, H, S, S)이다.
    # Casual Mask:(1, 1, S, S). Batch마다 casual들의 관계는 같다.(Batch 0에서도 미래는 못봄, Batch 1에서도 미래를 못봄...)
    # 쉽게 풀어서, 모든 batch에 동일한 mask를 사용할 것이기에 하나만을 사용할 것이다.
    # 따라서, Batch dimension은 1.
    # 모든 Head에서도 동일한 규칙을 가지고(하삼각 행렬) mask를 만들어서 사용할 것이다.
    # 따라서, Head dimension도 1이다.


    return mask

In [ ]:
# mask.py 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/mask.py

import torch


def create_padding_mask(
    token_ids,
    pad_idx,
):
    mask = token_ids != pad_idx

    mask = mask.unsqueeze(1).unsqueeze(2)

    return mask


def create_causal_mask(
    seq_len,
    device=None,
):
    mask = torch.ones(
        seq_len,
        seq_len,
        dtype=torch.bool,
        device=device,
    )

    mask = torch.tril(mask)

    mask = mask.unsqueeze(0).unsqueeze(0)

    return mask

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/mask.py


In [ ]:
# 저장 후 파일 구조 확인

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/mask.py

import torch


def create_padding_mask(
    token_ids,
    pad_idx,
):
    mask = token_ids != pad_idx

    mask = mask.unsqueeze(1).unsqueeze(2)

    return mask


def create_causal_mask(
    seq_len,
    device=None,
):
    mask = torch.ones(
        seq_len,
        seq_len,
        dtype=torch.bool,
        device=device,
    )

    mask = torch.tril(mask)

    mask = mask.unsqueeze(0).unsqueeze(0)

    return mask

Overwriting /content/drive/MyDrive/attention_is_all_you_need/src/mask.py


In [ ]:
# 저장 후 파일 구조 확인

for path in sorted(SRC_DIR.iterdir()):
    print(path.name)

__pycache__
attention.py
encoder.py
encoder_layer.py
feed_forward.py
mask.py
multi_head_attention.py
positional_encoding.py


In [ ]:
# mask 함수 import

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

print("Mask import 성공")

Mask import 성공


In [ ]:
# Padding Mask 생성

import torch

token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

pad_idx = 0

padding_mask = create_padding_mask(
    token_ids,
    pad_idx,
)

print(padding_mask)
print("Shape:", padding_mask.shape)
print("dtype:", padding_mask.dtype)

tensor([[[[ True,  True,  True,  True]]],


        [[[ True,  True, False, False]]]])
Shape: torch.Size([2, 1, 1, 4])
dtype: torch.bool


In [ ]:
# Padding Mask 기본 검증

assert padding_mask.shape == (
    2,
    1,
    1,
    4,
)

assert padding_mask.dtype == torch.bool

assert torch.equal(
    padding_mask[:, 0, 0, :],
    torch.tensor([
        [True, True, True, True],
        [True, True, False, False],
    ]),
)

print("Padding Mask 테스트 통과")

Padding Mask 테스트 통과


In [ ]:
# causal mask 생성

seq_len = 4

causal_mask = create_causal_mask(
    seq_len,
)

print(causal_mask)
print("Shape:", causal_mask.shape)
print("dtype:", causal_mask.dtype)

tensor([[[[ True, False, False, False],
          [ True,  True, False, False],
          [ True,  True,  True, False],
          [ True,  True,  True,  True]]]])
Shape: torch.Size([1, 1, 4, 4])
dtype: torch.bool


In [ ]:
# 4 * 4 causal mask 직접 보기

print(
    causal_mask[0, 0]
)

# 이때 출력값은 lower triangualr(아래가 0인 삼각행렬)이여야 한다.

tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])


In [ ]:
# causal mask 기본 검증

expected_causal_mask = torch.tensor([
    [True,  False, False, False],
    [True,  True,  False, False],
    [True,  True,  True,  False],
    [True,  True,  True,  True],
])

assert causal_mask.shape == (
    1,
    1,
    4,
    4,
)

assert causal_mask.dtype == torch.bool

assert torch.equal(
    causal_mask[0, 0],
    expected_causal_mask,
)

print("Causal Mask 테스트 통과")

Causal Mask 테스트 통과


In [ ]:
# Broadcasting 직접 확인

# 가짜 tensor 생성

batch_size = 2
num_heads = 3
query_len = 4
key_len = 4

scores = torch.randn(
    batch_size,
    num_heads,
    query_len,
    key_len,
)

print(
    "scores:",
    scores.shape,
)

print(
    "padding_mask:",
    padding_mask.shape,
)

print(
    "causal_mask:",
    causal_mask.shape,
)

# 실제 mask 연산

padding_applied = scores.masked_fill(
    ~padding_mask,
    float("-inf"),
)

causal_applied = scores.masked_fill(
    ~causal_mask,
    float("-inf"),
)

print(
    padding_applied.shape
)

print(
    causal_applied.shape
)

# mask를 repeat() 하지 않았는데도 성공함을 알 수 있음

scores: torch.Size([2, 3, 4, 4])
padding_mask: torch.Size([2, 1, 1, 4])
causal_mask: torch.Size([1, 1, 4, 4])
torch.Size([2, 3, 4, 4])
torch.Size([2, 3, 4, 4])


In [ ]:
# 실제 broadcast 결과 shape 확인

_, broadcast_padding_mask = (
    torch.broadcast_tensors(
        scores,
        padding_mask,
    )
)

_, broadcast_causal_mask = (
    torch.broadcast_tensors(
        scores,
        causal_mask,
    )
)

print(
    broadcast_padding_mask.shape
)

print(
    broadcast_causal_mask.shape
)

torch.Size([2, 3, 4, 4])
torch.Size([2, 3, 4, 4])


In [ ]:
# padding + causal 결합

decoder_token_ids = torch.tensor([
    [5, 8, 3, 0],
])

pad_idx = 0

decoder_padding_mask = (
    create_padding_mask(
        decoder_token_ids,
        pad_idx,
    )
)

decoder_causal_mask = (
    create_causal_mask(
        seq_len=4,
    )
)

combined_mask = (
    decoder_padding_mask
    & decoder_causal_mask
)

# shape 확인

print(
    "Padding:",
    decoder_padding_mask.shape,
)

print(
    "Causal:",
    decoder_causal_mask.shape,
)

print(
    "Combined:",
    combined_mask.shape,
)

Padding: torch.Size([1, 1, 1, 4])
Causal: torch.Size([1, 1, 4, 4])
Combined: torch.Size([1, 1, 4, 4])


In [ ]:
# 결합 Mask 내용 확인

print(
    "Padding Mask:"
)

print(
    decoder_padding_mask[0, 0, 0]
)

print()


print(
    "Causal Mask:"
)

print(
    decoder_causal_mask[0, 0]
)

print()


print(
    "Combined Mask:"
)

print(
    combined_mask[0, 0]
)

Padding Mask:
tensor([ True,  True,  True, False])

Causal Mask:
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])

Combined Mask:
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True, False]])


In [ ]:
# 기존 Multiheadattention에 padding mask 적용

from src.multi_head_attention import (
    MultiHeadAttention,
)

torch.manual_seed(0)

d_model = 8
num_heads = 2
seq_len = 4

attention = MultiHeadAttention(
    d_model,
    num_heads,
)

x = torch.randn(
    2,
    seq_len,
    d_model,
)

padding_mask = create_padding_mask(
    token_ids,
    pad_idx=0,
)

output, attention_weights = attention(
    x,
    x,
    x,
    padding_mask,
)

print(
    "Output:",
    output.shape,
)

print(
    "Attention:",
    attention_weights.shape,
)

Output: torch.Size([2, 4, 8])
Attention: torch.Size([2, 2, 4, 4])


In [ ]:
# pad key에 attention weight 확인

# 두번째 batch의 token은 7 2 PAD PAD 임으로 attention의 마지막 두 열은 전부 0이여야한다

print(
    attention_weights[
        1,
        :,
        :,
        :
    ]
)

# assert

assert torch.all(
    attention_weights[
        1,
        :,
        :,
        2:
    ] == 0
)

print(
    "Padding Mask Attention 테스트 통과"
)

tensor([[[0.5899, 0.4101, 0.0000, 0.0000],
         [0.2653, 0.7347, 0.0000, 0.0000],
         [0.6025, 0.3975, 0.0000, 0.0000],
         [0.3718, 0.6282, 0.0000, 0.0000]],

        [[0.4402, 0.5598, 0.0000, 0.0000],
         [0.4398, 0.5602, 0.0000, 0.0000],
         [0.7188, 0.2812, 0.0000, 0.0000],
         [0.4576, 0.5424, 0.0000, 0.0000]]], grad_fn=<SelectBackward0>)
Padding Mask Attention 테스트 통과


In [ ]:
'''
softmax의 연산 과정에서 -inf(-무한대)가 들어가기 때문에 softmax의 결과는 0으로 나오게 된다.
(수학적으로는 0으로 수렴)(메모리 적으로는 보통 -1e9를 넣는데, 이 경우 underflow로 인해 0.0으로 표기된다.)
'''

'\nsoftmax의 연산 과정에서 -inf(-무한대)가 들어가기 때문에 softmax의 결과는 0으로 나오게 된다.\n(수학적으로는 0으로 수렴)(메모리 적으로는 보통 -1e9를 넣는데, 이 경우 underflow로 인해 0.0으로 표기된다.)\n'

In [ ]:
# 기존 MultiHeadAttention에 Causal Mask 적용

# batch size = 1로 사용
torch.manual_seed(0)

x = torch.randn(
    1,
    4,
    d_model,
)

causal_mask = create_causal_mask(
    seq_len=4,
)

causal_output, causal_weights = (
    attention(
        x,
        x,
        x,
        causal_mask,
    )
)

print(
    "Output:",
    causal_output.shape,
)

print(
    "Attention:",
    causal_weights.shape,
)

Output: torch.Size([1, 4, 8])
Attention: torch.Size([1, 2, 4, 4])


In [ ]:
#  미래 위치 Attention이 0인지 확인

print(
    causal_weights[
        0,
        0
        ]
)

# assert

assert torch.all(
    causal_weights[
        :,
        :,
        future_positions,
    ] == 0
)

print(
    "Causal Mask Attention 테스트 통과"
)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3723, 0.6277, 0.0000, 0.0000],
        [0.3505, 0.3656, 0.2839, 0.0000],
        [0.1387, 0.2948, 0.3764, 0.1900]], grad_fn=<SelectBackward0>)
Causal Mask Attention 테스트 통과


In [ ]:
# Mask 적용 전후 attention shape 비교

x = torch.randn(
    2,
    4,
    d_model,
)

output_no_mask, _ = attention(
    x,
    x,
    x,
    None,
)

output_with_mask, _ = attention(
    x,
    x,
    x,
    padding_mask,
)

print(
    output_no_mask.shape
)

print(
    output_with_mask.shape
)

# 검증

assert (
    output_no_mask.shape
    ==
    output_with_mask.shape
)

print(
    "Mask 적용 전후 Attention Shape 유지"
)

In [ ]:
# Encoder에 Padding Mask 실제로 전달

from src.encoder import Encoder

torch.manual_seed(0)

batch_size = 2
seq_len = 4

d_model = 8
num_heads = 2
d_ff = 32
num_layers = 3

dropout = 0.0


token_ids = torch.tensor([ # shape = (1, 2, 4)
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

pad_idx = 0


padding_mask = create_padding_mask(
    token_ids,
    pad_idx,
)


x = torch.randn(
    batch_size,
    seq_len,
    d_model,
)


encoder = Encoder(
    d_model,
    num_heads,
    d_ff,
    num_layers,
    dropout,
)


output, attention_list = encoder(
    x,
    mask=padding_mask,
)


print(
    "Output:",
    output.shape,
)

print(
    "Number of attention tensors:",
    len(attention_list),
)

for i, weights in enumerate(
    attention_list
):
    print(
        f"Layer {i + 1}:",
        weights.shape,
    )

Output: torch.Size([2, 4, 8])
Number of attention tensors: 3
Layer 1: torch.Size([2, 2, 4, 4])
Layer 2: torch.Size([2, 2, 4, 4])
Layer 3: torch.Size([2, 2, 4, 4])


In [ ]:
# 모든 Encoder Layer의 PAD key weight 확인

# 두번째 batch에서 token index 2, 3는 PAD임 -> 모든 layer, head, query는 마지막 두 key의 위치가 0이여야함

for layer_index, weights in enumerate(
    attention_list
):
    pad_weights = weights[
        1,
        :,
        :,
        2:
    ]

    print(
        f"Layer {layer_index + 1}"
    )

    print(
        pad_weights
    )

    print(
        "All zero:",
        torch.all(
            pad_weights == 0
        ).item(),
    )

# 검증

for weights in attention_list:
    assert torch.all(
        weights[
            1,
            :,
            :,
            2:
        ] == 0
    )

print(
    "모든 Encoder Layer에서 "
    "PAD Key Attention 차단 확인"
)

Layer 1
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
All zero: True
Layer 2
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
All zero: True
Layer 3
tensor([[[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.],
         [0., 0.]]], grad_fn=<SliceBackward0>)
All zero: True
모든 Encoder Layer에서 PAD Key Attention 차단 확인


In [ ]:
# Encoder Mask 적용 전후 shape 비교

output_no_mask, _ = encoder(
    x,
    mask=None,
)

output_with_mask, _ = encoder(
    x,
    mask=padding_mask,
)

print(
    "No mask:",
    output_no_mask.shape,
)

print(
    "With mask:",
    output_with_mask.shape,
)

assert (
    output_no_mask.shape
    ==
    output_with_mask.shape
    ==
    (2, 4, 8)
)

print(
    "Encoder Shape 유지 확인"
)

No mask: torch.Size([2, 4, 8])
With mask: torch.Size([2, 4, 8])
Encoder Shape 유지 확인


In [ ]:
# Padding Mask batch / seq_len 변경 테스트

# 특정 형태(지금은 (2, 4))에 종속 되지는 않았는지 평가

padding_test_cases = [
    torch.tensor([
        [1, 2, 0],
    ]),

    torch.tensor([
        [1, 2, 3, 4],
        [5, 6, 0, 0],
    ]),

    torch.tensor([
        [1, 2, 3, 4, 5],
        [6, 7, 8, 0, 0],
        [9, 0, 0, 0, 0],
    ]),
]


for ids in padding_test_cases:
    mask = create_padding_mask(
        ids,
        pad_idx=0,
    )

    print(
        "Token IDs:",
        ids.shape,
        "→ Mask:",
        mask.shape,
    )

Token IDs: torch.Size([1, 3]) → Mask: torch.Size([1, 1, 1, 3])
Token IDs: torch.Size([2, 4]) → Mask: torch.Size([2, 1, 1, 4])
Token IDs: torch.Size([3, 5]) → Mask: torch.Size([3, 1, 1, 5])


In [ ]:
# Causal Mask에서 seq_len 변경 테스트

for seq_len in [
    1,
    3,
    5,
    7,
]:
    mask = create_causal_mask(
        seq_len
    )

    print(
        f"S={seq_len}",
        "→",
        mask.shape,
    )

S=1 → torch.Size([1, 1, 1, 1])
S=3 → torch.Size([1, 1, 3, 3])
S=5 → torch.Size([1, 1, 5, 5])
S=7 → torch.Size([1, 1, 7, 7])


In [ ]:
# 통합 테스트

import torch

from src.mask import (
    create_padding_mask,
    create_causal_mask,
)

from src.multi_head_attention import (
    MultiHeadAttention,
)

from src.encoder import Encoder


torch.manual_seed(0)


# -----------------------------------
# 1. Padding Mask
# -----------------------------------

token_ids = torch.tensor([
    [5, 8, 3, 9],
    [7, 2, 0, 0],
])

pad_idx = 0

padding_mask = create_padding_mask(
    token_ids,
    pad_idx,
)

assert padding_mask.shape == (
    2,
    1,
    1,
    4,
)

assert padding_mask.dtype == torch.bool

assert torch.equal(
    padding_mask[:, 0, 0, :],
    torch.tensor([
        [True, True, True, True],
        [True, True, False, False],
    ]),
)


# -----------------------------------
# 2. Causal Mask
# -----------------------------------

causal_mask = create_causal_mask(
    4
)

expected_causal_mask = torch.tensor([
    [True,  False, False, False],
    [True,  True,  False, False],
    [True,  True,  True,  False],
    [True,  True,  True,  True],
])

assert causal_mask.shape == (
    1,
    1,
    4,
    4,
)

assert causal_mask.dtype == torch.bool

assert torch.equal(
    causal_mask[0, 0],
    expected_causal_mask,
)


# -----------------------------------
# 3. Padding + Causal
# -----------------------------------

target_ids = torch.tensor([
    [5, 8, 3, 0],
])

target_padding_mask = (
    create_padding_mask(
        target_ids,
        pad_idx,
    )
)

target_causal_mask = (
    create_causal_mask(
        4
    )
)

combined_mask = (
    target_padding_mask
    & target_causal_mask
)

expected_combined_mask = torch.tensor([
    [True,  False, False, False],
    [True,  True,  False, False],
    [True,  True,  True,  False],
    [True,  True,  True,  False],
])

assert combined_mask.shape == (
    1,
    1,
    4,
    4,
)

assert torch.equal(
    combined_mask[0, 0],
    expected_combined_mask,
)


# -----------------------------------
# 4. Multi-Head Attention
# -----------------------------------

d_model = 8
num_heads = 2

attention = MultiHeadAttention(
    d_model,
    num_heads,
)

x = torch.randn(
    2,
    4,
    d_model,
)

output_no_mask, _ = attention(
    x,
    x,
    x,
    None,
)

output_with_mask, attention_weights = (
    attention(
        x,
        x,
        x,
        padding_mask,
    )
)

assert output_no_mask.shape == (
    2,
    4,
    8,
)

assert output_with_mask.shape == (
    2,
    4,
    8,
)

assert attention_weights.shape == (
    2,
    2,
    4,
    4,
)

assert torch.all(
    attention_weights[
        1,
        :,
        :,
        2:
    ] == 0
)


# -----------------------------------
# 5. Causal Attention
# -----------------------------------

causal_x = torch.randn(
    1,
    4,
    d_model,
)

_, causal_attention_weights = (
    attention(
        causal_x,
        causal_x,
        causal_x,
        causal_mask,
    )
)

future_positions = torch.triu(
    torch.ones(
        4,
        4,
        dtype=torch.bool,
    ),
    diagonal=1,
)

assert torch.all(
    causal_attention_weights[
        :,
        :,
        future_positions,
    ] == 0
)


# -----------------------------------
# 6. Encoder
# -----------------------------------

encoder = Encoder(
    d_model=8,
    num_heads=2,
    d_ff=32,
    num_layers=3,
    dropout=0.0,
)

encoder_x = torch.randn(
    2,
    4,
    8,
)

encoder_output, attention_list = (
    encoder(
        encoder_x,
        mask=padding_mask,
    )
)

assert encoder_output.shape == (
    2,
    4,
    8,
)

assert len(attention_list) == 3

for weights in attention_list:

    assert weights.shape == (
        2,
        2,
        4,
        4,
    )

    assert torch.all(
        weights[
            1,
            :,
            :,
            2:
        ] == 0
    )


# -----------------------------------
# 7. Shape change tests
# -----------------------------------

for batch_size, seq_len in [
    (1, 3),
    (2, 4),
    (3, 5),
]:
    ids = torch.ones(
        batch_size,
        seq_len,
        dtype=torch.long,
    )

    mask = create_padding_mask(
        ids,
        pad_idx=0,
    )

    assert mask.shape == (
        batch_size,
        1,
        1,
        seq_len,
    )


for seq_len in [
    1,
    3,
    5,
    7,
]:
    mask = create_causal_mask(
        seq_len
    )

    assert mask.shape == (
        1,
        1,
        seq_len,
        seq_len,
    )


print(
    "07) Mask 전체 통합 테스트 통과"
)

07) Mask 전체 통합 테스트 통과
